# Computing Similarities

In [1]:
from datasets import load_from_disk, Dataset
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, AutoModel
from datetime import datetime
from sentence_transformers import SentenceTransformer, util
import torch
import pickle
import os
from os.path import join
import pandas as pd
import re
import numpy as np
from secrets import randbits
from cupyx.scipy.spatial import distance
import sys
from config import ROOT_DIR

# PARAMS
cuda_device = "cuda"
default_save_path = join(ROOT_DIR,"datasets/multi_news/similarities/")
epsilons = [1] + [i for i in range(5, 101, 5)]
# END PARAMS
torch.set_default_device(cuda_device)

if not os.path.exists(default_save_path):
    os.makedirs(default_save_path)

def compute_similarities(embeddingsA:np.ndarray, embeddingsB:np.ndarray) -> list[int]:
    assert embeddingsA.shape[0] == embeddingsB.shape[0], "Inputs should have equal first dimension"
    similarities = []
    for i in range(embeddingsA.shape[0]):
        sim_tensor = util.cos_sim(embeddingsA[i], embeddingsB[i])
        similarities.append(sim_tensor.item())
    return similarities

def save_similarities(filename:str, similarities:list[int]) -> None:
    filepath = os.path.join(default_save_path, filename)
    with open(filepath, 'wb') as f:
        pickle.dump(similarities, f)

embedding_model = SentenceTransformer("all-mpnet-base-v2", device=cuda_device)

Load the original texts and encode them into embeddings.

In [2]:
mnews = load_from_disk(join(ROOT_DIR,"datasets/multi_news/concatenated_clean_1024_tokens"))
texts = mnews["document"]
texts_embedding = embedding_model.encode(texts)

## OG texts VS noisy texts
Compute the similarities between the original texts and sanitized texts

In [3]:
noisy_texts_folderpath = join(ROOT_DIR,"datasets/multi_news/noisy_texts/AsStrings/")

for epsilon in epsilons:
    with open(os.path.join(noisy_texts_folderpath, "epsi"+str(epsilon)+"full.pickle"), "rb") as f:
        noisy_texts = pickle.load(f)
    noisy_texts_embedding = embedding_model.encode(noisy_texts)
    similarities = compute_similarities(texts_embedding, noisy_texts_embedding)
    save_similarities("OGtextsVSnoisytexts_epsi"+str(epsilon)+"_Similarities.pickle", similarities)

## OG texts VS gen summaries
Compute the similarities between the original texts and the summaries generated from the original texts.

In [4]:
gensummary_filepath_per_model = {
    "bart": join(ROOT_DIR, "datasets/multi_news/bart_generated_summaries/OnOriginalTexts/full.pickle"),
    "Llama": join(ROOT_DIR, "datasets/multi_news/llama_generated_summaries/OnOriginalTexts/full.pickle"),
    "t5": join(ROOT_DIR, "datasets/multi_news/t5_generated_summaries/OnOriginalTexts/full.pickle"),
    "tinyllama": join(ROOT_DIR, "datasets/multi_news/tinyllamachat_generated_summaries/OnOriginalTexts/full.pickle")
}

for model_name, gensummaries_filepath in gensummary_filepath_per_model.items():
    with open(gensummaries_filepath, "rb") as f:
        gensummaries = pickle.load(f)
    
    gensummaries_embedding = embedding_model.encode(gensummaries)
    similarities = compute_similarities(texts_embedding, gensummaries_embedding)
    
    output_filename = f"OGtextVS{model_name}GensummarySimilarities.pickle"
    save_similarities(output_filename, similarities)

## OG text VS noisy gen summary
Compute the similarities between the original texts and the summaries generated from the sanitized texts.

In [5]:
noisygensummary_folderpath_per_model = {
    "bart": join(ROOT_DIR,"datasets/multi_news/bart_generated_summaries/OnSanitizedTexts"),
    "Llama": join(ROOT_DIR,"datasets/multi_news/llama_generated_summaries/OnSanitizedTexts"),
    "t5": join(ROOT_DIR,"datasets/multi_news/t5_generated_summaries/OnSanitizedTexts"),
    "tinyllama": join(ROOT_DIR,"datasets/multi_news/tinyllamachat_generated_summaries/OnSanitizedTexts"),
}

for model_name, noisygensummary_folderpath in noisygensummary_folderpath_per_model.items():
    for epsilon in epsilons:
        noisy_generated_summaries_filepath = join(noisygensummary_folderpath, f"epsi{epsilon}full.pickle")
        with open(noisy_generated_summaries_filepath, 'rb') as f:
            noisy_generated_summaries = pickle.load(f)
        
        noisy_generated_summaries_embedding = embedding_model.encode(noisy_generated_summaries)
        similarities = compute_similarities(texts_embedding, noisy_generated_summaries_embedding)

        output_filename = f"OGtextVS{model_name}noisygenSummaryEpsi{epsilon}Similarities.pickle"
        save_similarities(output_filename, similarities)